# 02 · Databricks Right check and dashboard table

This notebook ingests immutable Left events from a Unity Catalog Volume, then owns the Delta history, queue processing, and AI/BI dashboard.

Row lifecycle:

1. Notebook 01 reads local versioned configuration and writes an immutable Left event.
2. The local records folder is synced to the cloud prefix behind `RECORDS_PATH`.
3. This notebook idempotently merges the newest event per `check_id`, then fills its Right result.
4. A human may change only the latest row's `review_status` to `CHECKED_FAILED` or `CHECKED_SUCCEEDED`.
5. Local configuration uses `enabled: false` to stop future Left checks; Delta `review_status` remains human review state.

Connection JSON stores logical routing only. Credentials, tokens, passwords, and DSNs must remain in secret stores or environment variables.

The CREATE TABLE cell bootstraps a fresh table. It is not a schema migration for an older `count_watch_status` table.


In [ ]:
dbutils.widgets.text("TABLE_LIST", "<catalog>.<schema>.table_list")
dbutils.widgets.text("RECORDS_PATH", "/Volumes/<catalog>/<schema>/<volume>/count-watch")
dbutils.widgets.text("LOOKBACK_DAYS", "7")
TABLE_LIST = dbutils.widgets.get("TABLE_LIST")
RECORDS_PATH = dbutils.widgets.get("RECORDS_PATH").rstrip("/")
LOOKBACK_DAYS = int(dbutils.widgets.get("LOOKBACK_DAYS"))


In [ ]:
import re

_TABLE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*$")
if not _TABLE.fullmatch(TABLE_LIST):
    raise ValueError("TABLE_LIST must be catalog.schema.table")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_LIST} (
  check_id STRING NOT NULL,
  pair_name STRING NOT NULL,
  config_version BIGINT NOT NULL DEFAULT 1,
  query_date DATE,
  timezone STRING NOT NULL DEFAULT 'America/New_York',
  review_status STRING NOT NULL DEFAULT 'NOT_CHECKED',
  offset_days INT NOT NULL DEFAULT 0,
  aws_name STRING,
  left_query_variable STRING NOT NULL,
  right_query_variable STRING NOT NULL,
  left_conn_detail STRING NOT NULL,
  right_conn_detail STRING NOT NULL,
  left_partition STRING,
  right_partition STRING,
  left_count BIGINT,
  right_count BIGINT,
  count_state STRING NOT NULL DEFAULT 'NOT_RUN',
  left_checked_at TIMESTAMP,
  right_checked_at TIMESTAMP,
  error_message STRING,
  created_at TIMESTAMP NOT NULL DEFAULT current_timestamp(),
  updated_at TIMESTAMP NOT NULL DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.feature.allowColumnDefaults' = 'supported',
  'comment' = 'Pair configuration snapshots and latest-partition count history'
)
""")

constraints = {
    "valid_review_status": "review_status IN ('NOT_CHECKED', 'CHECKED_FAILED', 'CHECKED_SUCCEEDED')",
    "valid_count_state": "count_state IN ('NOT_RUN', 'PENDING_RIGHT', 'WAITING_RIGHT', 'SUCCESS', 'FAIL', 'ERROR')",
}
properties = {row.key: row.value for row in spark.sql(f"SHOW TBLPROPERTIES {TABLE_LIST}").collect()}
for name, condition in constraints.items():
    if f"delta.constraints.{name}" not in properties:
        spark.sql(f"ALTER TABLE {TABLE_LIST} ADD CONSTRAINT {name} CHECK ({condition})")


## Add or edit a pair

Insert a new pair as a configuration-only row. The JSON values are snapshots and are copied into every later historical row.

```sql
INSERT INTO <catalog>.<schema>.table_list VALUES (
  uuid(), 'accounts', 1, NULL, 'America/New_York',
  'NOT_CHECKED', 0, 'account-feed',
  'EFF_DT', 'dw_bus_dt',
  '{"source":"oracle","conn_name":"pcds","server_name":"oracle-a","table":"OWNER.ACCT_CURR","partition_type":"date"}',
  '{"source":"databricks","catalog":"main","schema":"curated","table":"accounts","partition_type":"date"}',
  NULL, NULL, NULL, NULL, 'NOT_RUN',
  NULL, NULL, NULL, current_timestamp(), current_timestamp()
);
```

To change configuration, append a new configuration-only row with a larger `config_version` so older check rows remain immutable. To change the human review state, update only the latest row for that pair. Keep the enum values exact:

- `review_status`: `NOT_CHECKED`, `CHECKED_FAILED`, `CHECKED_SUCCEEDED`
- `count_state`: `NOT_RUN`, `PENDING_RIGHT`, `WAITING_RIGHT`, `SUCCESS`, `FAIL`, `ERROR`


In [ ]:
import json
from datetime import UTC, date, datetime, timedelta

from delta.tables import DeltaTable
from pyspark.sql import Window, functions as F
from pyspark.sql import types as T

event_schema = T.StructType([
    T.StructField("schema_version", T.IntegerType(), False), T.StructField("event_id", T.StringType(), False),
    T.StructField("check_id", T.StringType(), False), T.StructField("pair_name", T.StringType(), False),
    T.StructField("config_version", T.LongType(), False), T.StructField("query_date", T.DateType(), False),
    T.StructField("timezone", T.StringType(), False), T.StructField("review_status", T.StringType(), False),
    T.StructField("offset_days", T.IntegerType(), False), T.StructField("aws_name", T.StringType(), True),
    T.StructField("left_query_variable", T.StringType(), False), T.StructField("right_query_variable", T.StringType(), False),
    T.StructField("left_conn_detail", T.StringType(), False), T.StructField("right_conn_detail", T.StringType(), False),
    T.StructField("left_partition", T.StringType(), True), T.StructField("left_count", T.LongType(), True),
    T.StructField("count_state", T.StringType(), False), T.StructField("left_checked_at", T.TimestampType(), False),
    T.StructField("error_message", T.StringType(), True), T.StructField("created_at", T.TimestampType(), False),
    T.StructField("updated_at", T.TimestampType(), False),
])
try:
    events = (spark.read.schema(event_schema).option("recursiveFileLookup", "true").json(RECORDS_PATH))
except Exception as exc:
    if "PATH_NOT_FOUND" not in str(exc):
        raise
    events = spark.createDataFrame([], event_schema)
latest = (events.withColumn("rn", F.row_number().over(Window.partitionBy("check_id").orderBy(F.col("left_checked_at").desc(), F.col("event_id").desc())))
          .where("rn = 1").drop("rn", "schema_version", "event_id"))
target = DeltaTable.forName(spark, TABLE_LIST)
insert_values = {column: f"source.{column}" for column in latest.columns}
insert_values.update({"right_partition": "NULL", "right_count": "NULL", "right_checked_at": "NULL"})
(target.alias("target").merge(latest.alias("source"), "target.check_id = source.check_id")
 .whenMatchedUpdate(condition="target.count_state = 'ERROR'", set={column: f"source.{column}" for column in latest.columns if column != "check_id"})
 .whenNotMatchedInsert(values=insert_values).execute())
print(f"Ingested {latest.count()} latest Left event(s) from {RECORDS_PATH}")

pending = spark.sql(f"""
WITH current_pair AS (
  SELECT pair_name, review_status,
         row_number() OVER (
           PARTITION BY pair_name
           ORDER BY config_version DESC, created_at DESC, check_id DESC
         ) AS rn
  FROM {TABLE_LIST}
)
SELECT pending.*
FROM {TABLE_LIST} pending
JOIN current_pair current
  ON pending.pair_name = current.pair_name AND current.rn = 1
WHERE current.review_status <> 'CHECKED_SUCCEEDED'
  AND pending.count_state = 'PENDING_RIGHT'
  AND pending.query_date >= date_sub(current_date(), {LOOKBACK_DAYS})
ORDER BY pending.query_date, pending.pair_name
""").collect()

print(f"{len(pending)} pending pair(s)")


In [ ]:
def parse_detail(value):
    detail = json.loads(value) if isinstance(value, str) else dict(value)
    forbidden = {"password", "token", "secret", "dsn"}

    def check_keys(item):
        if isinstance(item, dict):
            if any(str(key).lower() in forbidden for key in item):
                raise ValueError("Connection JSON must contain logical references only")
            for nested in item.values():
                check_keys(nested)
        elif isinstance(item, list):
            for nested in item:
                check_keys(nested)

    check_keys(detail)
    return detail


def table_name(detail):
    return ".".join(f"`{detail[key]}`" for key in ("catalog", "schema", "table"))


def partition_expr(variable, partition_type):
    return f"CAST({variable} AS DATE)" if partition_type == "date" else variable


def normalize(value, partition_type):
    if value is None:
        return None
    if partition_type == "date":
        return value.isoformat() if isinstance(value, date) else str(value)[:10]
    return str(value)


def expected_right_partition(left_partition, offset_days, partition_type):
    if partition_type == "date":
        return (date.fromisoformat(left_partition) + timedelta(days=offset_days)).isoformat()
    if partition_type in {"int", "numeric"}:
        return str(int(left_partition) + offset_days)
    if offset_days:
        raise ValueError("offset_days requires date, int, or numeric partition_type")
    return left_partition


def literal(value, partition_type):
    if partition_type in {"int", "numeric"}:
        return str(value)
    if partition_type == "date":
        return f"DATE '{str(value)[:10]}'"
    return "'" + str(value).replace("'", "''") + "'"


def right_latest(row):
    detail = parse_detail(row.right_conn_detail)
    if detail.get("source", "databricks") != "databricks":
        raise ValueError("Right source must be databricks")
    partition_type = detail.get("partition_type", "date")
    expression = partition_expr(row.right_query_variable, partition_type)
    filters = [detail["where"]] if detail.get("where") else []
    where = (" WHERE " + " AND ".join(f"({item})" for item in filters)) if filters else ""
    relation = table_name(detail)

    latest = spark.sql(f"SELECT MAX({expression}) AS value FROM {relation}{where}").first()["value"]
    if latest is None:
        return None, None, "WAITING_RIGHT", "latest Right partition is empty"

    latest_text = normalize(latest, partition_type)
    count_where = filters + [f"{expression} = {literal(latest_text, partition_type)}"]
    count_sql = f"SELECT COUNT(*) AS value FROM {relation} WHERE " + " AND ".join(f"({item})" for item in count_where)
    right_count = int(spark.sql(count_sql).first()["value"])
    expected = expected_right_partition(row.left_partition, int(row.offset_days or 0), partition_type)

    partition_match = latest_text == expected
    count_match = right_count == row.left_count
    state = "SUCCESS" if partition_match and count_match else "FAIL"
    reason = None if state == "SUCCESS" else (
        f"expected Right partition {expected}, found {latest_text}; "
        f"Left count {row.left_count}, Right count {right_count}"
    )
    return latest_text, right_count, state, reason


In [ ]:
result_schema = T.StructType([
    T.StructField("check_id", T.StringType(), False),
    T.StructField("right_partition", T.StringType(), True),
    T.StructField("right_count", T.LongType(), True),
    T.StructField("count_state", T.StringType(), False),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("right_checked_at", T.TimestampType(), False),
])

results = []
for row in pending:
    checked_at = datetime.now(UTC)
    try:
        right_partition, right_count, state, error = right_latest(row)
    except Exception as exc:
        right_partition, right_count, state = None, None, "ERROR"
        error = f"{type(exc).__name__}: {exc}"
    results.append((row.check_id, right_partition, right_count, state, error, checked_at))

if results:
    updates = spark.createDataFrame(results, result_schema)
    target = DeltaTable.forName(spark, TABLE_LIST)
    (
        target.alias("target")
        .merge(updates.alias("source"), "target.check_id = source.check_id")
        .whenMatchedUpdate(set={
            "right_partition": "source.right_partition",
            "right_count": "source.right_count",
            "count_state": "source.count_state",
            "error_message": "source.error_message",
            "right_checked_at": "source.right_checked_at",
            "updated_at": "source.right_checked_at",
        })
        .execute()
    )

display(spark.sql(f"""
SELECT query_date, pair_name, config_version, review_status, offset_days, aws_name,
       left_partition, right_partition, left_count, right_count, count_state,
       left_checked_at, right_checked_at, error_message
FROM {TABLE_LIST}
WHERE query_date >= date_sub(current_date(), {LOOKBACK_DAYS})
ORDER BY query_date DESC, pair_name
"""))


## Dashboard query

The AI/BI dashboard can read the table directly and expose a date filter:

```sql
SELECT query_date, pair_name, config_version, aws_name, offset_days,
       left_partition, right_partition, left_count, right_count,
       count_state, review_status, checked_at
FROM (
  SELECT *,
         greatest(left_checked_at, right_checked_at) AS checked_at
  FROM <catalog>.<schema>.table_list
)
WHERE query_date BETWEEN :start_date AND :end_date
ORDER BY query_date DESC, pair_name;
```

Keep connection JSON and query variables out of the default visualization. Users with appropriate dataset access can download the full table when those details are needed.
